# Talos — First GPU Training Run (tiny prototype)

This notebook trains the **Talos `tiny` prototype** (~254,272 parameters) for the
first time on a Google Colab GPU. It is the basic end-to-end acceptance test for
the training pipeline: it builds the canonical `tiny` config, trains it on a small
**fixed synthetic corpus**, and should show the loss falling from ~`log(vocab)`
(about 6.9 for vocab=1024) down to a low value, because the model **overfits** the
fixed corpus.

### Before you run it — one required setup

1. Open **Runtime → Change runtime type** in the Colab menu.
2. Set **Hardware accelerator** to **GPU** (T4 GPU is more than enough).
3. Click **Runtime → Run all** (or run the cells top-to-bottom).

**No installs required.** Google Colab already ships `torch` and `numpy`, which are
Talos's only runtime dependencies. The notebook never `pip install`s Talos itself —
it just clones the repo and runs the example scripts directly. The `tiny_train.py`
script auto-detects CUDA, so a GPU runtime is used automatically (with a CPU
fallback if you leave it on CPU runtime).

The whole thing takes well under a minute on a T4 GPU (and a minute or two on CPU).

In [ ]:
import torch
print("torch", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("no GPU detected — the run will fall back to CPU (still works, just slower)")

In [ ]:
# Clone the Talos repo (if not already present) and enter it.
# torch + numpy are already installed in Colab — nothing to pip install.
import os
import subprocess
import sys

if not os.path.isdir("talos"):
    subprocess.run(
        ["git", "clone", "https://github.com/Promethean-Studios/talos.git"],
        check=True,
    )
os.chdir("talos")
# os.chdir does NOT update sys.path in a notebook kernel, so add the repo
# root explicitly to make `from model import ...` work in the next cells.
sys.path.insert(0, os.getcwd())
print("working dir:", os.getcwd())

### Model summary

Confirm the canonical `tiny` preset loads and reports the expected parameter count
(254,272 parameters: vocab 1024, hidden 64, 2 layers, GQA with 4 query / 2 KV heads).

In [ ]:
import torch
from model import TalosGPT
from configs.presets import tiny_config

cfg = tiny_config().derive()
model = TalosGPT(cfg)
print(f"config: ffn_type={cfg.ffn_type}, hidden={cfg.hidden_size}, layers={cfg.num_layers}, "
      f"heads={cfg.num_attention_heads}, kv_heads={cfg.num_kv_heads}, vocab={cfg.vocab_size}")
print(f"trainable params: {model.num_parameters():,}")

### Train the tiny prototype on the Colab GPU

This runs the canonical `examples/tiny_train.py` via the Colab GPU. Meaningful flags:

* `--steps 300` — training steps (a few hundred is plenty to see the loss collapse).
* `--seq 64` — tokens per sequence.
* `--batch 8` — sequences per step.
* `--seed 0` — fixed seed, so the run is reproducible.

The command needs **no** `--device` flag: the script auto-detects CUDA and uses the
GPU. It prints one line per 25 steps and finally an `OK: ...` summary line.

In [ ]:
!python examples/tiny_train.py --steps 300 --seq 64 --batch 8 --seed 0

### What a successful first test looks like

✅ **SUCCESS** if the last cell ends with an `OK:` line whose loss falls clearly, e.g.:

```
OK: dense tiny (254272 params, device=cuda) loss 6.9170 -> 0.12XX over 300 steps (peak drop 6.8XXX)
```

* **Initial loss ≈ 6.9** — that is `log(vocab)` for vocab=1024, i.e. the model starts
  essentially guessing uniformly over the vocabulary.
* **Final loss well below 1** and falling — the training loop (forward + backward +
  AdamW optimizer) genuinely **works**: the model is learning the fixed corpus.
* `device=cuda` confirms it used the Colab GPU (it would say `device=cpu` on a CPU
  runtime — the same run still works).

The training target is a **fixed synthetic corpus** built from a low-order recurrence
(`x[t] = (x[t-2] + x[t-1]) mod vocab`), so the tiny model *overfits* it on purpose —
a deliberate, quickly-checkable way to prove the pipeline learns. This is **not**
real language data; pointing the pipeline at real data is a later, separate step.

### (Optional) Sanity-check inference: prefill + decode via the KV cache

Runs `examples/run_inference.py`, which prefills a random prompt in one forward pass
and then decodes one token at a time via the incremental KV cache — a quick check that
the generation path is alive. It uses a freshly built tiny model (the training script
above doesn't yet save checkpoints, which is out of scope for this basic test).

✅ SUCCESS if it ends with an `OK: generated 8 tokens: [...]` line.

In [ ]:
!python examples/run_inference.py --prompt_len 64 --decode 8 --seed 0

### Next steps

Once this first test is green, the natural next steps are:

1. Point training at **real data** produced by the Phase 3 data pipeline
   (`data/pipeline.py`), instead of the fixed synthetic corpus.
2. Log and save checkpoints, then load them back for inference on the trained model.
3. Establish a performance baseline and benchmark the prototype.

See `notebooks/README.md` in the repo for details.